# Step 1 - Price data pipeline check
Clones the repo, downloads the Nifty 200 price history and shows the data-quality tables.
Runs on Google Colab (Runtime > Run all). Takes a few minutes for the full download.

In [ ]:
import os, sys, subprocess
BRANCH = "claude/price-data-pipeline"   # change to "main" once merged
REPO = "https://github.com/jayantc14/stock_recommendation_tool"
if not os.path.exists("stock_recommendation_tool"):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO], check=True)
os.chdir("stock_recommendation_tool")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "pyarrow"], check=True)
# Editable installs made after the kernel started are not picked up by the running session
# (Colab), so put the package on the import path explicitly.
sys.path.insert(0, os.path.abspath("src"))


## Download (set LIMIT to a small number for a quick test, None for all 200)

In [ ]:
import pandas as pd
from stock_advisor.data.history_store import build_dataset, load_prices, suspect_moves
from stock_advisor.data.universe import BENCHMARK_TICKER, load_universe

LIMIT = None
tickers = [BENCHMARK_TICKER] + load_universe()["ticker"].tolist()[:LIMIT]
quality, errors = build_dataset(tickers, out_dir="data/prices")
print(f"{len(quality)}/{len(tickers)} downloaded; failures: {errors}")
quality.describe()

## Coverage - shortest histories

In [ ]:
quality.sort_values("rows").head(15)

## Suspect moves (|daily return| > 25%) - check for unadjusted splits and demergers

In [ ]:
suspects = pd.concat([suspect_moves(t, load_prices(t)) for t in quality["ticker"]], ignore_index=True)
suspects.loc[suspects["ret"].abs() > 0.4].sort_values("ret")

In [ ]:
import matplotlib.pyplot as plt
counts = pd.concat({t: load_prices(t)["Close"] for t in quality["ticker"]}, axis=1).notna().sum(axis=1)
counts.plot(figsize=(10, 3), title="Tickers with data per date"); plt.show()